In [1]:
from datasets import load_dataset

dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


In [2]:
example = dataset['train'][0]
for key, value in example.items():
    print(f"=={key}==")
    print(value)
    print()

==pubid==
21645374

==question==
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

==context==
{'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells i

In [3]:

def format_example(example):
    context = example['context']['contexts']
    context = " ".join(context)
    question = example['question']
    answer = example['final_decision']
    explanation = example['long_answer']

    format_text = (
        f"<instruction>: By looking to example answer the question\n"
        f"<context>: {context}\n"
        f"<question>: {question}\n"
        f"<answer>: {answer + ' ' + explanation}\n"
    )

    return {'text': format_text}
train_dataset = dataset['train'].map(format_example)
train_dataset[0]['text']

'<instruction>: By looking to example answer the question\n<context>: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage le

In [4]:
train_dataset = dataset['train'].map(format_example)

In [5]:
print(train_dataset[0]['text'])

<instruction>: By looking to example answer the question
<context>: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leav

In [6]:
split = train_dataset.train_test_split(test_size=0.1)

In [7]:
split

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'text'],
        num_rows: 900
    })
    test: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'text'],
        num_rows: 100
    })
})

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0')
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to(device)


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [22]:
def tokenize(example):
    tokens = tokenizer(
        example['text'],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    return tokens

train = split['train']
test = split['test']

tokenized_train = train.map(tokenize)
tokenized_test = test.map(tokenize)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [30]:
def add_label(example):
    example['labels'] = example['input_ids'].copy()
    return example

tokenized_train = tokenized_train.map(add_label)
tokenized_test = tokenized_test.map(add_label)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [33]:
from transformers import TrainingArguments, Trainer

trainings_arg = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_eval_batch_size=4,
    per_device_train_batch_size=4,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=trainings_arg,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

/opt/miniconda3/envs/finetune-medical/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 